# Self-guided lesson — Basic Exploratory Data Analysis (EDA)

Before cleaning or transforming a dataset aggressively, we need to understand what we have.

**Exploratory Data Analysis (EDA)** is the process of inspecting, summarizing, and visualizing data so that we can understand its structure, identify potential quality problems, and discover patterns worth investigating.

At this point in the course, the objective is deliberately modest. You will learn a **basic EDA workflow** that you can already use in the Data Wrangling Quest.

We will focus on:

- understanding what one row represents;
- inspecting columns, shapes, and data types;
- distinguishing numerical and categorical variables;
- checking missing values and duplicates;
- summarizing individual variables;
- using simple visualizations;
- exploring a few relationships between variables;
- validating observations before drawing conclusions.

More advanced statistical interpretation will come later in the course.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## 1. Load the data — then inspect before analyzing

We will use a dataset containing information about data-science jobs and salaries.

In [ ]:
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/data_science_salaries.csv"

df = pd.read_csv(url)

df.head()

Before calculating anything, ask:

> **What does one row represent?**

That question determines how every later count, average, grouping, or visualization should be interpreted.

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

These first checks tell us:

- how many rows and columns we have;
- which variables are available;
- the pandas data type assigned to each column;
- whether some columns contain missing values.

Remember that a pandas dtype is a **storage type**, not necessarily the conceptual meaning of the variable.

## 2. Numerical vs categorical variables

A useful first distinction is:

### Numerical variables

Values represent quantities for which arithmetic operations are meaningful.

Examples: salary, age, distance, number of purchases.

### Categorical variables

Values represent groups or labels.

Examples: country, job title, employment type.

A column containing numbers can still be categorical. A code such as `1 = Spain`, `2 = France`, `3 = Germany` should not be averaged just because pandas stores it as an integer.

Always interpret variables using their **meaning**, not only their dtype.

In [ ]:
numerical_cols = df.select_dtypes(include="number").columns
categorical_cols = df.select_dtypes(exclude="number").columns

print("Numerical columns:")
print(numerical_cols.tolist())

print("\nCategorical columns:")
print(categorical_cols.tolist())

This automatic selection is a useful starting point, but you should still inspect whether pandas' classification matches the meaning of each variable.

## 3. Basic data-quality checks

EDA and data cleaning are closely connected. Exploration often reveals the problems that need cleaning.

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
df.duplicated().sum()

In [ ]:
df.nunique().sort_values()

These checks help us identify:

- missing values;
- duplicated rows;
- columns with very few or very many unique values.

They do **not** tell us automatically what to do. A missing value, duplicate, or unusual category must be interpreted in context before changing the data.

## 4. Univariate analysis: one variable at a time

**Univariate analysis** means examining one variable.

For a numerical variable, useful questions include:

- What is a typical value?
- How spread out are the values?
- What are the minimum and maximum?
- Is the distribution roughly symmetric or strongly skewed?
- Are there unusually large or small observations?

In [ ]:
df["salary_in_usd"].describe()

In [ ]:
sns.histplot(data=df, x="salary_in_usd", bins=40)

plt.title("Distribution of salary in USD")
plt.show()

Do not rush from the histogram to a causal explanation. At this stage we can **describe what we observe**.

For example, we can look for concentration, spread, asymmetry, gaps, and extreme observations.

A useful discipline is:

**observation first → interpretation second**

### Categorical variables

For a categorical variable, frequencies and proportions are often more informative than numerical summary statistics.

In [ ]:
df["experience_level"].value_counts()

In [ ]:
df["experience_level"].value_counts(normalize=True)

In [ ]:
sns.countplot(data=df, x="experience_level")

plt.title("Number of records by experience level")
plt.show()

The table and chart answer essentially the same question in different forms: how the observations are distributed across categories.

## 5. Bivariate exploration: two variables

After understanding individual variables, we can begin asking whether two variables appear to be related.

At this stage, visualization is often enough to generate useful questions.

### Numerical variable vs categorical variable

A box plot can help us compare the distribution of a numerical variable across categories.

In [ ]:
sns.boxplot(
    data=df,
    x="experience_level",
    y="salary_in_usd"
)

plt.title("Salary in USD by experience level")
plt.show()

Ask descriptive questions:

- Do the groups appear to have different typical values?
- Which groups show more variability?
- Are there extreme observations?
- Do the distributions overlap?

A visible difference does not by itself prove that the category **causes** the difference.

### Numerical variable vs numerical variable

A scatter plot helps us inspect how two numerical variables vary together.

In [ ]:
sns.scatterplot(
    data=df,
    x="work_year",
    y="salary_in_usd",
    alpha=0.4
)

plt.title("Salary in USD by work year")
plt.show()

Look for patterns, clusters, unusual observations, and possible trends.

Later in the course you will study tools such as correlation and statistical inference in more depth. For now, the goal is to **see the data before trying to quantify every relationship**.

## 6. Grouping can answer focused questions

EDA becomes more useful when it is connected to a specific question.

For example:

> How does salary in USD differ across experience levels?

In [ ]:
salary_by_experience = (
    df.groupby("experience_level")["salary_in_usd"]
      .agg(["count", "mean", "median", "min", "max"])
)

salary_by_experience

Notice that we calculate both the **mean** and **median**. If a distribution contains very large values, the two may tell different stories about what is typical.

The purpose of the table is not to produce as many statistics as possible. It is to collect evidence relevant to a question.

## 7. Validate intermediate results

Suppose a category has a surprising salary. Before building a story around it, inspect the underlying rows.

In [ ]:
df.nlargest(
    5,
    "salary_in_usd"
)[["job_title", "experience_level", "employee_residence", "salary_in_usd"]]

This habit is extremely important:

**build → inspect → validate → continue**

A query or calculation can run successfully and still represent the wrong thing. Intermediate validation helps catch logical mistakes before they become conclusions.

# 💡 Check for understanding

Using the same dataset, investigate the question:

> **How do data-science job records differ by company size?**

Work incrementally:

1. Inspect the possible values of `company_size`.
2. Count the records in each company-size category.
3. Calculate the proportion represented by each category.
4. Visualize the category counts.
5. Compare `salary_in_usd` across company sizes using:
   - descriptive statistics;
   - an appropriate visualization.
6. Inspect any observations that appear unusually high or low.
7. Write **three descriptive observations** supported by your analysis.
8. For each observation, identify at least one limitation or question you would investigate before making a stronger claim.

Do not try to prove causality or perform a statistical test. The objective is to **understand and describe the data**.

# A reusable basic EDA workflow

When you receive a new dataset, a useful sequence is:

### 1. Understand
- What does one row represent?
- What variables are available?
- What do those variables mean?

### 2. Inspect
- Shape
- Column names
- Data types
- Missing values
- Duplicates
- Unique values

### 3. Describe individual variables
- Numerical → descriptive statistics + distribution
- Categorical → counts/proportions + bar chart

### 4. Explore relevant relationships
- Numerical vs categorical → grouped summaries + box plot
- Numerical vs numerical → scatter plot
- Categorical vs categorical → grouped counts or proportions when useful

### 5. Validate
Inspect the rows behind surprising results and check intermediate calculations.

### 6. Interpret carefully
Separate what you **observe** from what you think might **explain** it.

EDA is not a race to create every possible plot. Its purpose is to build enough understanding of the data to make better cleaning decisions and ask better analytical questions.